In [64]:
import pandas as pd
import numpy as np

def evaluate_signal_efficacy_extremes(df, 
                             start_year=2020, 
                             end_year=2024,
                             buys_path_template='../AI Portfolio Selection/buys_{}.csv',
                             sells_path_template='../AI Portfolio Selection/sells_{}.csv',
                             percentile_threshold=0.10):
    """
    Evaluates if LLM signals identify 'Extreme' stocks (Top OR Bottom performers),
    regardless of direction. Useful if the Quant agent has freedom to flip signs.
    
    *MODIFIED*: Enforces that firms must have 12 non-NaN months of return data.
    """
    
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    results = []
    
    # The random baseline is now double (Top% + Bottom%)
    baseline_chance = percentile_threshold * 2
    
    print(f"{'='*80}")
    print(f"SIGNAL EFFICACY ANALYSIS: EXTREMITY DETECTION ({start_year}-{end_year})")
    print(f"Target: Identify stocks that land in EITHER the Top {percentile_threshold:.0%} OR Bottom {percentile_threshold:.0%}")
    print(f"Constraint: Firms must have complete 12-month return data.")
    print(f"Baseline Random Chance: {baseline_chance:.0%}")
    print(f"{'='*80}")

    for year in range(start_year, end_year + 1):
        
        # --- 1. Load Signals ---
        try:
            buys_df = pd.read_csv(buys_path_template.format(year))
            sells_df = pd.read_csv(sells_path_template.format(year))
            
            buys = set(buys_df['permno'].values) if 'permno' in buys_df.columns else set(buys_df.index)
            sells = set(sells_df['permno'].values) if 'permno' in sells_df.columns else set(sells_df.index)
            
            common = buys.intersection(sells)
            buys = buys - common
            sells = sells - common
            
        except FileNotFoundError:
            print(f"⚠ Warning: Signals for {year} not found. Skipping.")
            continue
            
        if len(buys) == 0 and len(sells) == 0:
            continue

        # --- 2. Calculate Actual 1-Year Returns (STRICT) ---
        start_date = pd.Timestamp(f"{year-1}-12-01") 
        end_date = pd.Timestamp(f"{year}-11-30")
        
        year_data = df[(df['datadate'] >= start_date) & (df['datadate'] <= end_date)].copy()
        
        def calculate_annual_return(x):
            if x.count() < 12: # STRICT CHECK
                return np.nan
            return np.prod(1 + x) - 1

        annual_returns = year_data.groupby('permno')['ret_fwd_1'].apply(
            calculate_annual_return
        ).dropna()
        
        if annual_returns.empty:
            print(f"⚠ No complete return data found for {year}")
            continue

        # --- 3. Define Extremes (Top OR Bottom) ---
        n_firms = len(annual_returns)
        
        top_cutoff = annual_returns.quantile(1 - percentile_threshold)
        bottom_cutoff = annual_returns.quantile(percentile_threshold)
        
        true_winners = set(annual_returns[annual_returns >= top_cutoff].index)
        true_losers = set(annual_returns[annual_returns <= bottom_cutoff].index)
        
        true_extremes = true_winners.union(true_losers)
        
        # --- 4. Compare Signals to Extremes ---
        valid_buys = [p for p in buys if p in annual_returns.index]
        valid_sells = [p for p in sells if p in annual_returns.index]
        
        # Hits
        buys_extreme = [p for p in valid_buys if p in true_extremes]
        sells_extreme = [p for p in valid_sells if p in true_extremes]
        
        # Rates
        buy_hit_rate = len(buys_extreme) / len(valid_buys) if valid_buys else 0.0
        sell_hit_rate = len(sells_extreme) / len(valid_sells) if valid_sells else 0.0
        
        # Counts for Totals
        total_signals = len(valid_buys) + len(valid_sells)
        total_hits = len(buys_extreme) + len(sells_extreme)
        total_acc = total_hits / total_signals if total_signals > 0 else 0.0

        # --- 5. Store & Print ---
        res_row = {
            'Year': year,
            'N_Buys': len(valid_buys),
            'N_Sells': len(valid_sells),
            'Buy_Extremity_Rate': buy_hit_rate,
            'Sell_Extremity_Rate': sell_hit_rate,
            'Total_Accuracy': total_acc,
            'Total_Signals': total_signals,
            'Baseline_Chance': baseline_chance
        }
        results.append(res_row)
        
        print(f"[{year}] Universe: {n_firms}")
        print(f"  > Buys: {len(valid_buys)}. Extremity Rate: {buy_hit_rate:.1%} ({len(buys_extreme)} hits)")
        print(f"  > Sells: {len(valid_sells)}. Extremity Rate: {sell_hit_rate:.1%} ({len(sells_extreme)} hits)")
        print(f"  > Total accuracy: {total_acc}")

    # Create Summary DataFrame
    summary_df = pd.DataFrame(results)
    if not summary_df.empty:
        summary_df.set_index('Year', inplace=True)
        
        # Weighted Averages
        total_buys = summary_df['N_Buys'].sum()
        total_sells = summary_df['N_Sells'].sum()
        grand_total = total_buys + total_sells
        
        w_buy_acc = (summary_df['Buy_Extremity_Rate'] * summary_df['N_Buys']).sum() / total_buys if total_buys > 0 else 0
        w_sell_acc = (summary_df['Sell_Extremity_Rate'] * summary_df['N_Sells']).sum() / total_sells if total_sells > 0 else 0
        w_total_acc = (summary_df['Total_Accuracy'] * summary_df['Total_Signals']).sum() / grand_total if grand_total > 0 else 0
        
        print(f"\n{'='*80}")
        print("OVERALL EXTREMITY RESULTS (Strict Data)")
        print(f"{'='*80}")
        print(f"Total Signals:   {grand_total}")
        print(f"Buy Extremity:   {w_buy_acc:.1%} (vs {baseline_chance:.0%})")
        print(f"Sell Extremity:  {w_sell_acc:.1%} (vs {baseline_chance:.0%})")
        print(f"{'-'*40}")
        print(f"TOTAL WEIGHTED ACCURACY: {w_total_acc:.1%} (vs {baseline_chance:.0%})")
        
        lift = w_total_acc / baseline_chance if baseline_chance > 0 else 0
        print(f"Lift vs Random:  {lift:.2f}x")
        
        if w_total_acc > baseline_chance * 1.1:
             print("\n✅ CONCLUSION: The LLM successfully identifies high-information (extreme) stocks.")
        else:
             print("\n❌ CONCLUSION: The LLM signals are indistinguishable from random noise.")
             
    return summary_df

In [4]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [70]:
# ==========================================
# Example Usage
# ==========================================
evaluate_signal_efficacy_extremes(df, percentile_threshold=0.1)

SIGNAL EFFICACY ANALYSIS: EXTREMITY DETECTION (2020-2024)
Target: Identify stocks that land in EITHER the Top 10% OR Bottom 10%
Constraint: Firms must have complete 12-month return data.
Baseline Random Chance: 20%
[2020] Universe: 480
  > Buys: 28. Extremity Rate: 17.9% (5 hits)
  > Sells: 9. Extremity Rate: 44.4% (4 hits)
  > Total accuracy: 0.24324324324324326
[2021] Universe: 478
  > Buys: 3. Extremity Rate: 0.0% (0 hits)
  > Sells: 110. Extremity Rate: 25.5% (28 hits)
  > Total accuracy: 0.24778761061946902
[2022] Universe: 483
  > Buys: 21. Extremity Rate: 23.8% (5 hits)
  > Sells: 13. Extremity Rate: 23.1% (3 hits)
  > Total accuracy: 0.23529411764705882
[2023] Universe: 484
  > Buys: 60. Extremity Rate: 13.3% (8 hits)
  > Sells: 68. Extremity Rate: 35.3% (24 hits)
  > Total accuracy: 0.25
[2024] Universe: 483
  > Buys: 7. Extremity Rate: 28.6% (2 hits)
  > Sells: 258. Extremity Rate: 21.7% (56 hits)
  > Total accuracy: 0.2188679245283019

OVERALL EXTREMITY RESULTS (Strict Data)

,N_Buys,N_Sells,Buy_Extremity_Rate,Sell_Extremity_Rate,Total_Accuracy,Total_Signals,Baseline_Chance
Year,,,,,,,
2020,28,9,0.178571,0.444444,0.243243,37,0.2
2021,3,110,0.000000,0.254545,0.247788,113,0.2
2022,21,13,0.238095,0.230769,0.235294,34,0.2
2023,60,68,0.133333,0.352941,0.250000,128,0.2
2024,7,258,0.285714,0.217054,0.218868,265,0.2


In [59]:
import pandas as pd
import numpy as np

def evaluate_signal_efficacy_dynamic(df, 
                             start_year=2020, 
                             end_year=2024,
                             buys_path_template='../AI Portfolio Selection/buys_{}.csv',
                             sells_path_template='../AI Portfolio Selection/sells_{}.csv'):
    """
    Evaluates LLM signal accuracy using dynamic thresholds.
    Checks if N buys -> Top N returns, and M sells -> Bottom M returns.
    Includes Overall Accuracy metric.
    
    *MODIFIED*: Enforces that firms must have 12 non-NaN months of return data 
    to be included in the 'Actual Return' calculation.
    """
    
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    results = []
    
    print(f"{'='*80}")
    print(f"SIGNAL EFFICACY ANALYSIS: DYNAMIC THRESHOLDS ({start_year}-{end_year})")
    print(f"Standard: If N buys are generated, how many are in the actual Top N?")
    print(f"Constraint: Firms must have complete 12-month return data.")
    print(f"{'='*80}")

    for year in range(start_year, end_year + 1):
        
        # --- 1. Load Signals ---
        try:
            buys_df = pd.read_csv(buys_path_template.format(year))
            sells_df = pd.read_csv(sells_path_template.format(year))
            
            # Extract Permnos
            buys = set(buys_df['permno'].values) if 'permno' in buys_df.columns else set(buys_df.index)
            sells = set(sells_df['permno'].values) if 'permno' in sells_df.columns else set(sells_df.index)
            
            # Remove overlaps
            common = buys.intersection(sells)
            buys = buys - common
            sells = sells - common
            
        except FileNotFoundError:
            print(f"⚠ Warning: Signals for {year} not found. Skipping.")
            continue
            
        if len(buys) == 0 and len(sells) == 0:
            continue

        # --- 2. Calculate Actual 1-Year Returns for the Universe ---
        start_date = pd.Timestamp(f"{year-1}-12-01") 
        end_date = pd.Timestamp(f"{year}-11-30")
        
        year_data = df[(df['datadate'] >= start_date) & (df['datadate'] <= end_date)].copy()
        
        # --- DATA QUALITY CHECK ---
        # We calculate the product of returns ONLY if we have 12 observations.
        # This prevents comparing a 1-month return (e.g. delisted firm) against 12-month returns.
        
        def calculate_annual_return(x):
            if x.count() < 12: # STRICT CHECK: Must have 12 valid months
                return np.nan
            return np.prod(1 + x) - 1

        annual_returns = year_data.groupby('permno')['ret_fwd_1'].apply(
            calculate_annual_return
        ).dropna().sort_values(ascending=False)
        
        if annual_returns.empty:
            print(f"⚠ No complete 12-month return data found for {year}")
            continue

        total_universe = len(annual_returns)
        
        # Filter signals to only those that exist in the COMPLETE return data
        valid_buys = [p for p in buys if p in annual_returns.index]
        valid_sells = [p for p in sells if p in annual_returns.index]
        
        n_buys = len(valid_buys)
        n_sells = len(valid_sells)
        
        if n_buys == 0 and n_sells == 0:
            print(f"[{year}] No valid signals matched with complete return data.")
            continue

        # --- 3. Define Dynamic Truth (Top N and Bottom M) ---
        actual_top_n = set(annual_returns.head(n_buys).index) if n_buys > 0 else set()
        actual_bottom_m = set(annual_returns.tail(n_sells).index) if n_sells > 0 else set()
        
        # --- 4. Validation Calculation ---
        
        # Buys
        correct_buys = [p for p in valid_buys if p in actual_top_n]
        buy_acc = len(correct_buys) / n_buys if n_buys > 0 else 0.0
        
        # Sells
        correct_sells = [p for p in valid_sells if p in actual_bottom_m]
        sell_acc = len(correct_sells) / n_sells if n_sells > 0 else 0.0
        
        # Total Accuracy
        total_signals = n_buys + n_sells
        total_correct = len(correct_buys) + len(correct_sells)
        total_acc = total_correct / total_signals if total_signals > 0 else 0.0
        
        # Baselines (Random Chance)
        buy_baseline = n_buys / total_universe if total_universe > 0 else 0
        sell_baseline = n_sells / total_universe if total_universe > 0 else 0
        
        # Weighted Baseline
        expected_random_correct = (n_buys * buy_baseline) + (n_sells * sell_baseline)
        total_baseline = expected_random_correct / total_signals if total_signals > 0 else 0

        # --- 5. Store & Print ---
        res_row = {
            'Year': year,
            'N_Buys': n_buys,
            'N_Sells': n_sells,
            'Buy_Precision': buy_acc,
            'Sell_Precision': sell_acc,
            'Total_Accuracy': total_acc,
            'Buy_Baseline': buy_baseline,
            'Sell_Baseline': sell_baseline,
            'Total_Baseline': total_baseline,
            'Expected_Random': expected_random_correct, # Store for weighted calc
            'Total_Signals': total_signals # Store for weighted calc
        }
        results.append(res_row)
        
        print(f"[{year}] Universe (Complete Data): {total_universe}")
        print(f"  > BUYS:  {len(correct_buys)}/{n_buys} correct ({buy_acc:.1%}). Random: {buy_baseline:.1%}")
        print(f"  > SELLS: {len(correct_sells)}/{n_sells} correct ({sell_acc:.1%}). Random: {sell_baseline:.1%}")
        print(f"  > TOTAL: {total_correct}/{total_signals} correct ({total_acc:.1%}). Random: {total_baseline:.1%}")

    # Create Summary DataFrame
    summary_df = pd.DataFrame(results)
    if not summary_df.empty:
        summary_df.set_index('Year', inplace=True)
        
        # Weighted Averages
        total_buys = summary_df['N_Buys'].sum()
        total_sells = summary_df['N_Sells'].sum()
        grand_total = total_buys + total_sells
        
        w_buy_acc = (summary_df['Buy_Precision'] * summary_df['N_Buys']).sum() / total_buys if total_buys > 0 else 0
        w_sell_acc = (summary_df['Sell_Precision'] * summary_df['N_Sells']).sum() / total_sells if total_sells > 0 else 0
        w_total_acc = (summary_df['Total_Accuracy'] * summary_df['Total_Signals']).sum() / grand_total if grand_total > 0 else 0
        
        # CORRECT WEIGHTED BASELINE CALCULATION
        w_total_base = summary_df['Expected_Random'].sum() / grand_total if grand_total > 0 else 0

        print(f"\n{'='*80}")
        print("OVERALL DYNAMIC PRECISION RESULTS")
        print(f"{'='*80}")
        print(f"Total Signals Analyzed: {grand_total}")
        print(f"Overall Buy Precision:  {w_buy_acc:.1%}")
        print(f"Overall Sell Precision: {w_sell_acc:.1%}")
        print(f"{'-'*40}")
        print(f"OVERALL ACCURACY:       {w_total_acc:.1%} (Weighted Baseline: {w_total_base:.1%})")
        
        lift = w_total_acc / w_total_base if w_total_base > 0 else 0
        print(f"Lift vs Random:         {lift:.2f}x")

    return summary_df

In [60]:
evaluate_signal_efficacy_dynamic(df)

SIGNAL EFFICACY ANALYSIS: DYNAMIC THRESHOLDS (2020-2024)
Standard: If N buys are generated, how many are in the actual Top N?
Constraint: Firms must have complete 12-month return data.
[2020] Universe (Complete Data): 480
  > BUYS:  0/28 correct (0.0%). Random: 5.8%
  > SELLS: 0/9 correct (0.0%). Random: 1.9%
  > TOTAL: 0/37 correct (0.0%). Random: 4.9%
[2021] Universe (Complete Data): 478
  > BUYS:  0/3 correct (0.0%). Random: 0.6%
  > SELLS: 19/110 correct (17.3%). Random: 23.0%
  > TOTAL: 19/113 correct (16.8%). Random: 22.4%
[2022] Universe (Complete Data): 483
  > BUYS:  0/21 correct (0.0%). Random: 4.3%
  > SELLS: 0/13 correct (0.0%). Random: 2.7%
  > TOTAL: 0/34 correct (0.0%). Random: 3.7%
[2023] Universe (Complete Data): 484
  > BUYS:  1/60 correct (1.7%). Random: 12.4%
  > SELLS: 9/68 correct (13.2%). Random: 14.0%
  > TOTAL: 10/128 correct (7.8%). Random: 13.3%
[2024] Universe (Complete Data): 483
  > BUYS:  0/7 correct (0.0%). Random: 1.4%
  > SELLS: 146/258 correct (56.6%)

,N_Buys,N_Sells,Buy_Precision,Sell_Precision,Total_Accuracy,Buy_Baseline,Sell_Baseline,Total_Baseline,Expected_Random,Total_Signals
Year,,,,,,,,,,
2020,28,9,0.000000,0.000000,0.000000,0.058333,0.018750,0.048705,1.802083,37
2021,3,110,0.000000,0.172727,0.168142,0.006276,0.230126,0.224183,25.332636,113
2022,21,13,0.000000,0.000000,0.000000,0.043478,0.026915,0.037145,1.262940,34
2023,60,68,0.016667,0.132353,0.078125,0.123967,0.140496,0.132748,16.991736,128
2024,7,258,0.000000,0.565891,0.550943,0.014493,0.534161,0.520434,137.915114,265
